# Performance per change

Reads `results.jsonl` written by `npm run perf` (see `README.md`). Each row is one recorded scenario of one run.

Frame times are measured with vsync off, so they reflect actual cost rather than the display refresh rate. GPU times come from `EXT_disjoint_timer_query_webgl2` and are indicative only: on macOS they go through ANGLE/Metal and can overlap between frames.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.json_normalize(pd.read_json('results.jsonl', lines=True).to_dict('records'))
df['timestamp'] = pd.to_datetime(df['timestamp'])

# labels in the order they were first benchmarked
label_order = df.groupby('label')['timestamp'].min().sort_values().index.tolist()
df['label'] = pd.Categorical(df['label'], categories=label_order, ordered=True)

print('machines:', df['machine.gpu'].unique().tolist())
df[['timestamp', 'label', 'sha', 'dirty', 'scenario', 'run', 'fps', 'frameMs.p50', 'frameMs.p95', 'gpuMs.p50', 'drawCalls']].tail(12)

Only compare numbers from the same machine and browser. Set `MACHINE` to pick one if there is more than one.

In [ ]:
MACHINE = df['machine.gpu'].iloc[-1]
d = df[df['machine.gpu'] == MACHINE]

# median over runs, per change and scenario
summary = (d.groupby(['label', 'scenario'], observed=True)
             [['fps', 'frameMs.p50', 'frameMs.p95', 'frameMs.p99', 'cpuMs.p50', 'gpuMs.p50', 'drawCalls', 'vertices', 'heapMB']]
             .median()
             .round(2))
summary

In [ ]:
metrics = [('frameMs.p50', 'frame time p50 (ms)'), ('frameMs.p95', 'frame time p95 (ms)'),
           ('gpuMs.p50', 'GPU time p50 (ms)'), ('drawCalls', 'draw calls / frame')]
scenarios = sorted(d['scenario'].unique())

fig, axes = plt.subplots(len(metrics), 1, figsize=(11, 3.2 * len(metrics)), sharex=True)
for ax, (col, title) in zip(axes, metrics):
    pivot = d.groupby(['label', 'scenario'], observed=True)[col].median().unstack('scenario')
    pivot.plot(ax=ax, marker='o')
    # spread between runs, so noise is visible next to real changes
    for s in scenarios:
        runs = d[d['scenario'] == s].groupby('label', observed=True)[col]
        ax.fill_between(range(len(pivot)), runs.min().reindex(pivot.index), runs.max().reindex(pivot.index), alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.grid(alpha=0.3)
axes[-1].set_xticks(range(len(pivot)))
axes[-1].set_xticklabels(pivot.index, rotation=20, ha='right')
plt.tight_layout()

Change relative to the first label (the baseline), per scenario. Positive is slower.

In [ ]:
base = summary.xs(label_order[0], level='label')
delta = (summary[['frameMs.p50', 'frameMs.p95', 'gpuMs.p50', 'drawCalls']]
         .sub(base[['frameMs.p50', 'frameMs.p95', 'gpuMs.p50', 'drawCalls']], level='scenario')
         .div(base[['frameMs.p50', 'frameMs.p95', 'gpuMs.p50', 'drawCalls']], level='scenario') * 100).round(1)
delta.add_suffix(' Δ%')